# LaunchReady: quickstart audit of Qwen/Qwen3-4B on Kaggle T4 x2

**Does the official model-card quickstart work as written on a free GPU?** docs/roadmap.md
Phase 3 asks us to act as a first-time user: copy the card's snippets, run them exactly as
written, and log every break in `docs/friction_log.md`.

| Audit | Snippet (read live from the card at a recorded commit) | Packages |
|---|---|---|
| **1a** | The Python block under "## Quickstart" (Transformers, `torch_dtype="auto"`, `device_map="auto"`, `max_new_tokens=32768`, thinking on) | Kaggle's preinstalled torch + transformers, as a Kaggle user would have them |
| **1b** | The `vllm serve Qwen/Qwen3-4B ...` line under "## Quickstart" | `vllm==0.32.0` (the card asks for `vllm>=0.8.5`; same pin as notebook 01) |

Nothing in the snippets is edited. Each runs in a child process with a time limit; its full
output goes to `results/quickstart_audit/`, and the notebook prints a draft friction-log
entry for anything that breaks.

**To run:** Kaggle → Session options → Accelerator **GPU T4 x2**, Internet **on** →
*Save Version → Save & Run All* (~20–60 min, depending on how long the model thinks in 1a).


## Environment

In [ ]:
import json
import platform
import re
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path

REPO_ID = "Qwen/Qwen3-4B"
VLLM_PIN = "vllm==0.32.0"
OUT_DIR = Path("results/quickstart_audit")
OUT_DIR.mkdir(parents=True, exist_ok=True)


def run(cmd: list[str], timeout: int | None = None) -> str:
    return subprocess.run(cmd, capture_output=True, text=True, check=True, timeout=timeout).stdout


try:
    smi = run(["nvidia-smi"])
except (FileNotFoundError, subprocess.CalledProcessError) as e:
    raise RuntimeError(
        "No NVIDIA GPU visible. Session options → Accelerator → 'GPU T4 x2', then Run all again."
    ) from e
print(smi)

# Package versions are read in a child process so this kernel never touches CUDA.
VERSIONS = """
import importlib.metadata as md, json, sys
out = {"python": sys.version.split()[0]}
for pkg in ("torch", "transformers", "accelerate", "vllm"):
    try:
        out[pkg] = md.version(pkg)
    except md.PackageNotFoundError:
        out[pkg] = None
print(json.dumps(out))
"""


def package_versions() -> dict:
    return json.loads(run([sys.executable, "-c", VERSIONS]).strip().splitlines()[-1])


gpus = run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"]
).strip()
ENV = {
    "gpu": gpus.splitlines()[0].split(",")[0].strip(),
    "gpu_count": len(gpus.splitlines()),
    "nvidia_smi": gpus.splitlines(),
    "driver_cuda": (re.search(r"CUDA Version:\s*([\d.]+)", smi) or [None, None])[1],
    "platform": platform.platform(),
    "date_utc": datetime.now(UTC).isoformat(timespec="seconds"),
    "packages_kaggle_stock": package_versions(),
}
print(json.dumps(ENV, indent=2))

## The card, at a recorded commit

In [ ]:
# Read the card at a fixed commit and pull out the two snippets verbatim.
from huggingface_hub import HfApi, hf_hub_download

CARD_SHA = HfApi().model_info(REPO_ID).sha
card = Path(hf_hub_download(REPO_ID, "README.md", revision=CARD_SHA)).read_text()
(OUT_DIR / "card_README.md").write_text(card)
quickstart = card.split("## Quickstart", 1)[1].split("\n## ", 1)[0]

python_block = re.search(r"```python\n(.*?)```", quickstart, re.S)
vllm_line = re.search(r"^\s*(vllm serve [^\n]+)$", quickstart, re.M)
if not python_block or not vllm_line:
    raise RuntimeError("The card's Quickstart section changed shape; look at card_README.md")
SNIPPET_1A = python_block.group(1)
COMMAND_1B = vllm_line.group(1).strip()
(OUT_DIR / "1a_snippet.py").write_text(SNIPPET_1A)
(OUT_DIR / "1b_command.sh").write_text(COMMAND_1B + "\n")
ENV["card_sha"] = CARD_SHA
print(f"card {REPO_ID} @ {CARD_SHA}\n\n--- 1a ---\n{SNIPPET_1A}\n--- 1b ---\n{COMMAND_1B}")

In [ ]:
import os
import signal
import time
import urllib.request

ERROR_LINE = re.compile(r"Error|Exception|Traceback|error:|FAILED|Killed", re.I)
RESULTS = {}


def first_errors(text: str, n: int = 12) -> list[str]:
    return [line for line in text.splitlines() if ERROR_LINE.search(line)][:n]


def tail(text: str, n: int = 25) -> list[str]:
    return text.strip().splitlines()[-n:]


def stop_group(proc: subprocess.Popen) -> None:
    """SIGTERM the process group (server + workers), then SIGKILL stragglers."""
    for sig, wait in ((signal.SIGTERM, 60), (signal.SIGKILL, 30)):
        try:
            os.killpg(proc.pid, sig)
        except ProcessLookupError:
            return
        try:
            proc.wait(timeout=wait)
            return
        except subprocess.TimeoutExpired:
            continue


def wait_gpus_free(timeout: int = 180) -> None:
    deadline = time.time() + timeout
    while time.time() < deadline:
        used = run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"])
        if max(int(x) for x in used.split()) < 1024:
            return
        time.sleep(3)
    print("warning: GPU memory not released; the next audit may be affected")


def record(name: str, command: str, log_path: Path, outcome: str, seconds: float, **extra) -> None:
    text = log_path.read_text(errors="replace")
    RESULTS[name] = {
        "audit": name,
        "command": command,
        "outcome": outcome,
        "seconds": round(seconds, 1),
        "first_error_lines": first_errors(text),
        "last_lines": tail(text),
        "log": str(log_path),
        **extra,
    }
    print(f"{name}: {outcome} in {seconds:.0f}s · log {log_path}")
    for line in RESULTS[name]["first_error_lines"]:
        print("   ", line[:200])

## Audit 1a: Transformers quickstart, Kaggle's stock packages

Run exactly as written. `max_new_tokens=32768` with thinking on can take a while on a T4, so the
run is capped at 40 minutes; hitting the cap is recorded as its own outcome.


In [ ]:
TIMEOUT_1A = 2400
log_1a = OUT_DIR / "1a_transformers.log"
start = time.time()
with log_1a.open("w") as log:
    try:
        proc = subprocess.run(
            [sys.executable, str(OUT_DIR / "1a_snippet.py")],
            stdout=log,
            stderr=subprocess.STDOUT,
            timeout=TIMEOUT_1A,
        )
        outcome = "worked" if proc.returncode == 0 else f"failed (exit {proc.returncode})"
    except subprocess.TimeoutExpired:
        outcome = f"timed out after {TIMEOUT_1A}s"
output = log_1a.read_text(errors="replace")
answered = "content:" in output and bool(output.split("content:")[-1].strip())
record(
    "1a",
    f"python 1a_snippet.py  # card {REPO_ID} @ {CARD_SHA[:7]}, Quickstart Python block",
    log_1a,
    outcome if outcome != "worked" or answered else "ran but printed no answer",
    time.time() - start,
    packages=ENV["packages_kaggle_stock"],
)
wait_gpus_free()

## Audit 1b: vLLM one-liner

Installs the pinned vLLM (which brings its own torch and transformers), then runs the card's
`vllm serve` line exactly as written. If the server comes up, one chat request checks that the
reasoning parser splits thinking from the answer, then the server is shut down.


In [ ]:
pip = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--progress-bar", "off", VLLM_PIN],
    capture_output=True,
    text=True,
)
if pip.returncode:
    print(pip.stdout[-3000:], pip.stderr[-3000:])
    raise RuntimeError(f"pip install {VLLM_PIN} failed (see above). Is Internet on?")
ENV["packages_vllm"] = package_versions()
print(json.dumps(ENV["packages_vllm"], indent=2))

TIMEOUT_1B = 1800
log_1b = OUT_DIR / "1b_vllm.log"
start = time.time()
with log_1b.open("w") as log:
    proc = subprocess.Popen(
        ["bash", "-c", COMMAND_1B], stdout=log, stderr=subprocess.STDOUT, start_new_session=True
    )


def healthy() -> bool:
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2) as r:
            return r.status == 200
    except Exception:
        return False


reply = None
try:
    while time.time() - start < TIMEOUT_1B and proc.poll() is None and not healthy():
        time.sleep(5)
    if healthy():
        body = {
            "model": REPO_ID,
            "messages": [{"role": "user", "content": "How many r's are in strawberry?"}],
            "max_tokens": 1024,
        }
        request = urllib.request.Request(
            "http://127.0.0.1:8000/v1/chat/completions",
            data=json.dumps(body).encode(),
            headers={"Content-Type": "application/json"},
        )
        try:
            with urllib.request.urlopen(request, timeout=600) as r:
                reply = json.load(r)["choices"][0]["message"]
        except Exception as e:
            reply = {"error": f"{type(e).__name__}: {e}"}
finally:
    seconds = time.time() - start
    exit_code = proc.poll()
    stop_group(proc)

if reply and "error" not in reply:
    outcome = "worked" if reply.get("content") else "served but returned no content"
    has_reasoning = bool(reply.get("reasoning_content") or reply.get("reasoning"))
elif reply:
    outcome, has_reasoning = f"served but the chat request failed: {reply['error']}", False
elif exit_code is not None:
    outcome, has_reasoning = f"failed (exit {exit_code})", False
else:
    outcome, has_reasoning = f"not healthy after {TIMEOUT_1B}s", False
record("1b", COMMAND_1B, log_1b, outcome, seconds, packages=ENV["packages_vllm"],
       reasoning_separated=has_reasoning, reply=reply)  # fmt: skip
wait_gpus_free()

## Summary and draft friction-log entries

Saved to `results/quickstart_audit/kaggle_summary.json` with the environment. For each break,
a draft entry in the friction-log template (7.3) is printed; root cause and fix still need a
human read of the log.


In [ ]:
(OUT_DIR / "kaggle_summary.json").write_text(
    json.dumps({"environment": ENV, "audits": RESULTS}, indent=2, default=str) + "\n"
)
for name, r in RESULTS.items():
    print(f"{name}: {r['outcome']}")
print(f"\nwrote {OUT_DIR / 'kaggle_summary.json'}\n")

for name, r in RESULTS.items():
    packages = ", ".join(f"{k} {v}" for k, v in r["packages"].items() if v)
    when = (
        f"{ENV['date_utc'][:10]} · {ENV['gpu_count']}× {ENV['gpu']} (Kaggle) · "
        f"driver {ENV['nvidia_smi'][0].split(',')[-1].strip()} · CUDA {ENV['driver_cuda']} · "
        f"{packages} · card {REPO_ID} @ {CARD_SHA[:7]}"
    )
    if r["outcome"] == "worked":
        print(
            f"- Audit {name} worked as written: {when} · `{r['command']}` ({r['seconds']:.0f}s)\n"
        )
        continue
    excerpt = " · ".join(f"`{line.strip()[:160]}`" for line in r["first_error_lines"][:4])
    print(
        f"### <title> (quickstart audit {name})\n"
        f"- When / where: {when}\n"
        f"- Exact command: `{r['command']}`\n"
        f"- What happened (error excerpt): {r['outcome']}. {excerpt}\n"
        "- Root cause:\n- Fix or workaround:\n- Severity:\n- Filed:\n"
        "- Requirement this implies:\n"
    )

## After the run

Download `results/quickstart_audit/` from the version's **Output** tab and copy it into the repo
next to the Mac audit files. Breaks go into `docs/friction_log.md` (template 7.3) before
anything is changed; don't file issues or PRs on the model's or vLLM's repos without the
project owner's OK.
